# SwitchLingua: weeks 4 to 6 pipeline (Team 10)

**Predictive Multitask Learning for Streaming Code-Switching**

This notebook contains all the code for weeks 4 to 6 in one place:

| Week | Milestone | Sections |
|---|---|---|
| 4 | Clean text and map language IDs | 1 to 4 |
| 5 | Build switch and duration labels | 5 to 7 |
| 6 | Streaming loader and baselines | 8 to 11 |

**Task.** At every token *t*, using only tokens 1..*t*, predict `y_sw` (does token *t+1* switch language?) and `y_dur` (length of the new segment: 1-2, 3-6 or 7+ tokens).

**How to run.** Top to bottom. In Colab, first add a secret named `HF_TOKEN` (key icon in the sidebar) with a Hugging Face token whose account has accepted the dataset terms. Runtime on a normal CPU: about 15 to 20 minutes, mostly section 6.

The code is the same as in the team repository (`src/` and `scripts/`), copied here so the notebook runs on its own. Results match `results/runs/logreg_val_seed13` in the repo.

In [ ]:
# In Colab, install the one package that is not preinstalled:
#   !pip install -q wordfreq
import os, time, json, math
from pathlib import Path
from collections import Counter

# Some Windows conda setups load two OpenMP runtimes (torch + numpy); this lets them coexist.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import pandas as pd

SEED = 13
WORK = Path("switchlingua_work")          # all outputs of this notebook go here
WORK.mkdir(exist_ok=True)

try:  # Colab: read the Hugging Face token from the secret, never paste it in a cell
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # elsewhere: use `huggingface-cli login` or a local cache
print("outputs in", WORK.resolve())

# Week 4: clean text and map language IDs

## 1. Load and flatten the dataset

The text column (`data_generation_result`) is a stringified Python object written by an LLM. Most rows are lists of sentences, but some are dicts (`{'topic', 'instances'}`, `{'speaker', 'message'}`), nested lists, or plain strings. The parser below handles all of them. The dataset version is pinned so every teammate gets the same rows in the same order.

In [ ]:
# ---- from src/data.py ----
"""Load SwitchLingua_text and flatten it into one row per sentence.

The `data_generation_result` column is a stringified Python object written by an
LLM generator. Most rows are a list of sentence strings, but some rows (mostly
Cantonese multi-turn) hold dicts such as {"topic": ..., "instances": [...]} or
{"speaker": ..., "message": ...}, nested lists, or a bare string. `extract_sentences`
handles all of these and keeps only the text fields.
"""


import ast
import os
from dataclasses import dataclass

import pandas as pd

DATASET_ID = "Shelton1013/SwitchLingua_text"
# Pinned dataset version. The splits in results/splits identify rows by position, so every
# machine must load the same version in the same order. Change only together with new splits.
DATASET_REVISION = "6e048d27e3b2cb3a7a926288d4915745a0ab2801"

# Dict keys that hold metadata, not utterance text.
METADATA_KEYS = {"topic", "speaker", "role", "name", "id", "turn"}

# Sentences longer than this (whitespace words) are generator artifacts, e.g. a whole
# conversation pasted into one string. They are flagged, not silently dropped.
MAX_WORDS = 120

META_COLUMNS = [
    "topic", "tense", "perspective", "cs_ratio", "gender", "age", "education_level",
    "first_language", "second_language", "conversation_type", "cs_function", "cs_type", "score",
]


def pair_name(first_language: str, second_language: str) -> str:
    return f"{first_language}-{second_language}"


def load_raw_dataset(columns: list[str] | None = None):
    """The train split as a memory-mapped Hugging Face Dataset, with a `row_id` column.

    Use this with `iter_raw_batches` to process the data in batches.
    """
    from datasets import load_dataset

    ds = load_dataset(DATASET_ID, split="train", revision=DATASET_REVISION)
    ds = ds.select_columns(columns or META_COLUMNS + ["data_generation_result"])
    return ds.add_column("row_id", list(range(len(ds))))


def iter_raw_batches(ds, batch_size: int):
    """Yield pandas DataFrames of `batch_size` rows from a dataset made by `load_raw_dataset`."""
    for start in range(0, len(ds), batch_size):
        yield ds.select(range(start, min(start + batch_size, len(ds)))).to_pandas()


def load_raw() -> pd.DataFrame:
    """Load the whole train split into pandas."""
    df = load_raw_dataset().to_pandas()
    return df[["row_id"] + [c for c in df.columns if c != "row_id"]]


def extract_sentences(obj) -> list[str]:
    """Recursively collect sentence strings from a parsed generation result."""
    if isinstance(obj, str):
        return [obj]
    if isinstance(obj, (list, tuple)):
        return [s for item in obj for s in extract_sentences(item)]
    if isinstance(obj, dict):
        return [s for k, v in obj.items() if k not in METADATA_KEYS for s in extract_sentences(v)]
    return []


def parse_generation_result(raw: str) -> list[str]:
    """Parse one `data_generation_result` value into a list of sentences.

    Values that are not valid Python literals are plain sentences (91 Cantonese rows).
    """
    try:
        obj = ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        obj = raw
    return [s.strip() for s in extract_sentences(obj)]


@dataclass
class FlattenReport:
    rows_in: int
    sentences_out: int
    dropped_empty: pd.Series
    dropped_too_long: pd.Series
    duplicate_sentences: pd.Series

    def __str__(self) -> str:
        lines = [f"rows in: {self.rows_in}, sentences out: {self.sentences_out}"]
        for name in ("dropped_empty", "dropped_too_long", "duplicate_sentences"):
            counts = getattr(self, name)
            lines.append(f"{name}: {int(counts.sum())} " + (str(counts.to_dict()) if counts.sum() else ""))
        return "\n".join(lines)


def flatten(df: pd.DataFrame) -> tuple[pd.DataFrame, FlattenReport]:
    """One row per sentence, with the source row's metadata and a `pair` column.

    Empty and over-long sentences are removed and counted per pair in the report.
    Duplicate sentences are kept but counted, so the split step can keep them on one side.
    """
    meta = [c for c in df.columns if c != "data_generation_result"]
    records = []
    for row in df.itertuples(index=False):
        r = row._asdict()
        for i, sent in enumerate(parse_generation_result(r["data_generation_result"])):
            records.append({**{c: r[c] for c in meta}, "sent_idx": i, "text": sent})
    out = pd.DataFrame.from_records(records)
    out.insert(1, "pair", [pair_name(a, b) for a, b in zip(out.first_language, out.second_language)])

    n_words = out.text.str.split().str.len()
    empty = n_words == 0
    too_long = n_words > MAX_WORDS
    report = FlattenReport(
        rows_in=len(df),
        sentences_out=int((~empty & ~too_long).sum()),
        dropped_empty=out[empty].pair.value_counts(),
        dropped_too_long=out[too_long].pair.value_counts(),
        duplicate_sentences=out[~empty & ~too_long & out.text.duplicated()].pair.value_counts(),
    )
    out = out[~empty & ~too_long].reset_index(drop=True)
    return out, report

In [ ]:
ds = load_raw_dataset()
print(ds)
pairs = pd.Series([pair_name(a, b) for a, b in zip(ds["first_language"], ds["second_language"])])
print(pairs.value_counts().to_string())

## 2. Tokenization

One tokenization shared by every model, so all models are scored on the same targets:
- words are split on whitespace and punctuation (`didn't`, `week-end` stay whole);
- **each Chinese character is one token** (the mBERT rule). Treating a whole run of Chinese characters as one token would leak the future: the run only "ends" when a non-Chinese character arrives, which is often the switch itself;
- hiragana and katakana runs are separate tokens; digits form one token.

In [ ]:
# ---- from src/tokens.py ----
"""Model-independent word tokenization shared by all models.

Labels are defined on these tokens, so XLM-R and XGLM are scored on identical targets
whatever their subword vocabularies are.

Rules:
- Letters and combining marks of one script form a word ("didn't", "week-end" stay whole:
  an apostrophe or hyphen between two letters of the same script is kept inside).
- Every Han character is its own token (the mBERT rule), because Chinese, Cantonese and
  Japanese do not mark word boundaries with spaces.
- Hiragana and katakana runs are separate tokens ("しい", "メンバー").
- A run of digits is one token of script "num".
- Punctuation, symbols, emoji and whitespace are not tokens. They are kept in the
  original text and recovered from the character offsets.
"""


import unicodedata
from dataclasses import dataclass

JOINERS = {"'", "’", "-", "‐"}


@dataclass(frozen=True)
class Token:
    text: str
    start: int  # character offset in the sentence
    end: int
    script: str


def char_script(ch: str) -> str | None:
    """Script of a letter or mark, "num" for digits, None for everything else."""
    cat = unicodedata.category(ch)
    if cat == "Nd":
        return "num"
    if not (cat.startswith("L") or cat.startswith("M")):
        return None
    cp = ord(ch)
    if 0x4E00 <= cp <= 0x9FFF or 0x3400 <= cp <= 0x4DBF or 0xF900 <= cp <= 0xFAFF or 0x20000 <= cp <= 0x2EBEF:
        return "han"
    if 0x3040 <= cp <= 0x309F:
        return "hiragana"
    if 0x30A0 <= cp <= 0x30FF or 0x31F0 <= cp <= 0x31FF or 0xFF66 <= cp <= 0xFF9F:
        return "katakana"
    if 0xAC00 <= cp <= 0xD7AF or 0x1100 <= cp <= 0x11FF or 0x3130 <= cp <= 0x318F:
        return "hangul"
    if 0x0600 <= cp <= 0x06FF or 0x0750 <= cp <= 0x077F or 0xFB50 <= cp <= 0xFDFF or 0xFE70 <= cp <= 0xFEFF:
        return "arabic"
    if 0x0900 <= cp <= 0x097F:
        return "devanagari"
    if 0x0400 <= cp <= 0x052F:
        return "cyrillic"
    if cp <= 0x024F or 0x1E00 <= cp <= 0x1EFF or 0xFF21 <= cp <= 0xFF5A:
        return "latin"
    if cat.startswith("M"):
        return "mark"  # combining mark outside the blocks above, attaches to its word
    return "other"


def tokenize(text: str) -> list[Token]:
    tokens: list[Token] = []
    start = None
    script = None

    def close(end: int) -> None:
        nonlocal start, script
        if start is not None:
            tokens.append(Token(text[start:end], start, end, script))
        start, script = None, None

    for i, ch in enumerate(text):
        s = char_script(ch)
        if s == "mark" and start is not None:
            continue
        if s == "han":
            close(i)
            tokens.append(Token(ch, i, i + 1, "han"))
        elif s is None:
            nxt = char_script(text[i + 1]) if i + 1 < len(text) else None
            if ch in JOINERS and start is not None and script not in ("num",) and nxt == script:
                continue
            close(i)
        elif s == script:
            continue
        else:
            close(i)
            start, script = i, s
    close(len(text))
    return tokens

In [ ]:
for s in ["It's week-end, right?", "聽日係個big day, 我哋", "新しいメンバー", "भारत के institutes"]:
    print(s, "->", [t.text for t in tokenize(s)])

## 3. Romanized Hindi word list

56% of Hindi-English rows are written entirely in Latin script ("Logon ko acting bahut pasand aayi"). To separate Hindi from English there, we derive a word list from the corpus: Latin words inside **Devanagari** rows are English insertions, so they show how English looks in this text. Words frequent in romanized rows but almost absent from that reference are Hindi.

In [ ]:
from wordfreq import zipf_frequency

ROMAN_HINDI_PATH = WORK / "roman_hindi.tsv"
MIN_COUNT = 5      # words seen fewer times in romanized rows are not listed
HINDI_SCORE = 1.0  # at least 10x more frequent in romanized rows -> Hindi
ENGLISH_SCORE = 0.3  # at or below this -> English (not written to the file)
ENGLISH_ZIPF = 3.5   # in the middle zone, this English frequency or more -> English
SHORT = 3            # in the middle zone, short common English words -> context


def build_roman_hindi_lexicon(ds) -> pd.DataFrame:
    # Count Latin words in romanized vs Devanagari Hindi-English rows, then classify each word.
    roman, reference = Counter(), Counter()
    hindi = ds.filter(lambda x: x == "Hindi", input_columns="first_language")
    for raw in hindi["data_generation_result"]:
        toks = [t for sent in parse_generation_result(raw) for t in tokenize(sent)]
        latin = [t.text.lower() for t in toks if t.script == "latin"]
        (reference if any(t.script == "devanagari" for t in toks) else roman).update(latin)
    n_roman, n_ref = sum(roman.values()), sum(reference.values())
    rows = []
    for word, c in roman.items():
        if c < MIN_COUNT:
            continue
        score = math.log10((c + 1) / n_roman) - math.log10((reference[word] + 1) / n_ref)
        z_en = zipf_frequency(word, "en")
        if score >= HINDI_SCORE:
            cls = "hindi"
        elif score <= ENGLISH_SCORE:
            continue
        elif z_en < ENGLISH_ZIPF:
            cls = "hindi"
        else:
            cls = "context" if len(word) <= SHORT else "english"
        rows.append((word, c, reference[word], score, z_en, cls))
    out = pd.DataFrame(rows, columns=["word", "count_roman", "count_reference", "score", "zipf_en", "class"])
    return out.sort_values("count_roman", ascending=False).reset_index(drop=True)


lexicon = build_roman_hindi_lexicon(ds)
lexicon.to_csv(ROMAN_HINDI_PATH, sep="\t", index=False, float_format="%.2f")
print(lexicon["class"].value_counts().to_dict())
lexicon.head(15)

## 4. Language ID per token

Each token gets one of its row's two languages:
1. **Script rule**: if the token's alphabet belongs to only one of the two languages, that decides it (fully covers Arabic, Cantonese, Chinese, Hindi in Devanagari, Japanese, Korean, Russian).
2. **Frequency rule** for the 8 Latin-script pairs: compare the word's frequency in each language (`wordfreq`). Romanized Hindi rows use the word list from section 3.
3. **Context rule**: undecided tokens (names, numbers, words common to both languages) take the language of the nearest decided neighbor, preferring the left one.

Labels may use the whole sentence; only the model **inputs** must be prefix-only.

In [ ]:
# ---- from src/lid.py ----
"""Token-level language identification (LID) for SwitchLingua sentences.

The dataset gives only the two languages of each row, not a language per word, so we
derive one. Every token is tagged with one of the row's two languages:

1. Script rule: if the token's script belongs to only one of the two languages
   (Devanagari in Hindi-English, Latin in Cantonese-English), that decides it.
   This covers 7 of the 15 pairs completely.
2. Frequency rule (both languages written in Latin script): compare the word's Zipf
   frequency in each language with `wordfreq`. A clear winner (gap >= MARGIN) decides it.
   Romanized Hindi: in Hindi-English rows written without any Devanagari, Hindi words are in
   Latin script too. There a corpus-derived word list (src/resources/roman_hindi.tsv, built by
   scripts/build_roman_hindi_lexicon.py) decides, and other words count as English when
   they are common in English (Zipf >= ROMAN_EN_ZIPF).
3. Context rule: tokens left undecided (numbers, names, words common in both languages
   such as "no" or "me" in Spanish-English) take the language of the nearest decided
   token, preferring the left neighbor when the two sides disagree.

Labels may use the whole sentence (including the right context in rule 3). Only the
model's input features must be prefix-only.
"""


import csv
from functools import lru_cache
from pathlib import Path

from wordfreq import zipf_frequency


LANG_SCRIPTS = {
    "Arabic": {"arabic"},
    "Cantonese": {"han"},
    "Chinese": {"han"},
    "Hindi": {"devanagari"},
    "Japanese": {"han", "hiragana", "katakana"},
    "Korean": {"hangul"},
    "Russian": {"cyrillic"},
    "English": {"latin"},
    "French": {"latin"},
    "German": {"latin"},
    "Italian": {"latin"},
    "Malay": {"latin"},
    "Philippines": {"latin"},
    "Spanish": {"latin"},
    "Vietnamese": {"latin"},
}

WORDFREQ_CODE = {
    "English": "en", "French": "fr", "German": "de", "Italian": "it",
    "Malay": "ms", "Philippines": "fil", "Spanish": "es", "Vietnamese": "vi",
}

MARGIN = 0.3    # Zipf gap needed for the frequency rule. Kept small because the Malay,
                # Filipino and other web corpora contain many English loanwords, so English
                # words are often only 0.3 to 1.5 Zipf higher in English than in the other list.
MIN_ZIPF = 1.0  # below this in both languages the word is unknown (likely a name)

# ROMAN_HINDI_PATH is set in section 3 (built from the corpus)
ROMAN_EN_ZIPF = 3.0  # unlisted word in a romanized Hindi row: English if at least this common

# How each label was decided, kept for diagnostics.
SCRIPT, FREQ, CONTEXT, DEFAULT = "script", "freq", "context", "default"


@lru_cache(maxsize=500_000)
def _zipf(word: str, code: str) -> float:
    return zipf_frequency(word, code)


@lru_cache(maxsize=1)
def roman_hindi_lexicon() -> dict[str, str]:
    """word -> "hindi", "english" or "context"."""
    with ROMAN_HINDI_PATH.open(encoding="utf-8", newline="") as f:
        return {r["word"]: r["class"] for r in csv.DictReader(f, delimiter="	")}


def is_romanized_hindi(tokens: list[Token], lang1: str) -> bool:
    """True for Hindi-English text with no Devanagari at all. Pass all tokens of a row."""
    return lang1 == "Hindi" and not any(t.script == "devanagari" for t in tokens)


def _tag_roman_hindi(tok: Token) -> tuple[str | None, str | None]:
    if tok.script != "latin":
        return None, None
    word = tok.text.lower()
    cls = roman_hindi_lexicon().get(word)
    if cls == "hindi":
        return "Hindi", FREQ
    if cls == "english" or (cls is None and _zipf(word, "en") >= ROMAN_EN_ZIPF):
        return "English", FREQ
    return None, None


def tag_token(tok: Token, lang1: str, lang2: str, romanized: bool = False) -> tuple[str | None, str | None]:
    """Language of one token by the script or frequency rule, or (None, None)."""
    if romanized:
        return _tag_roman_hindi(tok)
    in1 = tok.script in LANG_SCRIPTS[lang1]
    in2 = tok.script in LANG_SCRIPTS[lang2]
    if in1 != in2:
        return (lang1 if in1 else lang2), SCRIPT
    if in1 and in2 and tok.script == "latin":
        word = tok.text.lower()
        z1 = _zipf(word, WORDFREQ_CODE[lang1])
        z2 = _zipf(word, WORDFREQ_CODE[lang2])
        if max(z1, z2) >= MIN_ZIPF and abs(z1 - z2) >= MARGIN:
            return (lang1 if z1 > z2 else lang2), FREQ
    return None, None


def tag_sentence(
    tokens: list[Token], lang1: str, lang2: str, romanized: bool = False
) -> tuple[list[str], list[str]]:
    """Language label and decision source for every token in a sentence.

    `romanized` must be decided per row with `is_romanized_hindi`, not per sentence:
    an all-English sentence inside a Devanagari row is ordinary English.
    """
    tagged = [tag_token(t, lang1, lang2, romanized) for t in tokens]
    langs = [lang for lang, _ in tagged]
    sources = [src for _, src in tagged]

    decided = [i for i, lang in enumerate(langs) if lang is not None]
    if not decided:
        return [lang1] * len(tokens), [DEFAULT] * len(tokens)

    out_langs, out_sources = list(langs), list(sources)
    for i, lang in enumerate(langs):
        if lang is not None:
            continue
        left = next((langs[j] for j in range(i - 1, -1, -1) if langs[j] is not None), None)
        right = next((langs[j] for j in range(i + 1, len(langs)) if langs[j] is not None), None)
        out_langs[i] = left if left is not None else right
        out_sources[i] = CONTEXT
    return out_langs, out_sources

In [ ]:
# Hand-checked examples (the same as the repository's unit tests)
def show(text, l1, l2, romanized=False):
    toks = tokenize(text)
    langs, _ = tag_sentence(toks, l1, l2, romanized)
    return [f"{t.text}/{lang[:2]}" for t, lang in zip(toks, langs)]

assert show("الموضوع معقد شوي، you know?", "Arabic", "English") == ["الموضوع/Ar", "معقد/Ar", "شوي/Ar", "you/En", "know/En"]
assert show("呢個IG update好正", "Cantonese", "English") == ["呢/Ca", "個/Ca", "IG/En", "update/En", "好/Ca", "正/Ca"]
assert show("Yo no sé, but I think so", "Spanish", "English") == ["Yo/Sp", "no/Sp", "sé/Sp", "but/En", "I/En", "think/En", "so/En"]
assert show("Logon ko acting bahut pasand aayi, the buzz was amazing", "Hindi", "English", romanized=True) == [
    "Logon/Hi", "ko/Hi", "acting/En", "bahut/Hi", "pasand/Hi", "aayi/Hi", "the/En", "buzz/En", "was/En", "amazing/En"]
assert show("यह main reason है", "Hindi", "English") == ["यह/Hi", "main/En", "reason/En", "है/Hi"]
for ex in [("Saya rasa public should use fragrance macam tu", "Malay", "English"),
           ("Das kostet 50 euros, mais c'est cher", "German", "French")]:
    print(show(*ex))
print("language ID checks passed")

# Week 5: switch and duration labels

## 5. Labels

At token *t*: `y_sw = 1` if token *t+1* is in a different language; `y_dur` = length bin of the segment starting at *t+1* (only where `y_sw = 1`). The last token has no *t+1* and gets no target (`-1`, ignored).

In [ ]:
# ---- from src/labels.py ----
"""Predictive targets built from per-token language tags.

At token t (0-based), using the true tags of the whole sentence:
- y_sw[t]  = 1 if lang[t+1] != lang[t], else 0.
- y_dur[t] = length bin of the segment that starts at t+1, only where y_sw[t] = 1.

The last token has no t+1, so both targets are IGNORE there. y_dur is IGNORE wherever
y_sw is not 1, which matches CrossEntropyLoss(ignore_index=-1) and the rule that
duration is only evaluated at true switch points.

A segment that runs to the end of the sentence is counted as is (it may be cut short
by the sentence boundary). This is a known right-censoring effect to mention in the report.
"""


IGNORE = -1

# (upper bound inclusive, class id): 1-2 tokens -> 0, 3-6 -> 1, 7+ -> 2
DURATION_BINS = ((2, 0), (6, 1))
DURATION_NAMES = ("1-2", "3-6", "7+")


def duration_class(length: int) -> int:
    for upper, cls in DURATION_BINS:
        if length <= upper:
            return cls
    return len(DURATION_BINS)


def segment_length_from(langs: list[str], start: int) -> int:
    """Number of consecutive tokens with the same language starting at `start`."""
    end = start
    while end < len(langs) and langs[end] == langs[start]:
        end += 1
    return end - start


def make_labels(langs: list[str]) -> tuple[list[int], list[int]]:
    """Return (y_sw, y_dur) for one sentence, one value per token."""
    n = len(langs)
    y_sw = [IGNORE] * n
    y_dur = [IGNORE] * n
    for t in range(n - 1):
        switch = langs[t + 1] != langs[t]
        y_sw[t] = int(switch)
        if switch:
            y_dur[t] = duration_class(segment_length_from(langs, t + 1))
    return y_sw, y_dur

In [ ]:
# Hand-checked examples (same as the repository's unit tests)
X = IGNORE
assert make_labels(["es", "es", "es", "en", "en", "en", "en"]) == ([0, 0, 1, 0, 0, 0, X], [X, X, 1, X, X, X, X])
assert make_labels(["ar", "ar", "ar", "en", "en"]) == ([0, 0, 1, 0, X], [X, X, 0, X, X])
assert make_labels(["yue", "yue", "en", "en", "yue", "yue"]) == ([0, 1, 0, 1, 0, X], [X, 0, X, 0, X, X])
assert make_labels(["es"] + ["en"] * 8)[1][0] == 2
assert make_labels(["en"]) == ([X], [X])
print("label checks passed")

## 6. Splits and the full processing pass

- **80 / 10 / 10** train / val / test **by conversation**, so sentences of one conversation never cross splits.
- **Stratified** by pair, with Hindi split by script (16 strata).
- Conversations that share an identical sentence are kept in the same split, so no sentence appears in both train and test.

Then every sentence is tokenized, tagged and labeled, in batches of 20,000 rows written to `switchlingua_work/processed/` (the full data does not fit comfortably in 8 GB of RAM at once).

In [ ]:
# ---- from src/splits.py ----
"""Fixed train/val/test splits at the source-row (conversation) level.

- The unit is the source row, so sentences of one conversation never sit in two splits.
- Rows that share any identical sentence text are grouped together (union-find), so a
  duplicated sentence cannot appear in both train and test.
- Stratified by `strata` (the language pair, with Hindi-English split by script), so each
  stratum gets the same train/val/test proportions.
"""


import numpy as np
import pandas as pd

SPLIT_FRACTIONS = {"train": 0.8, "val": 0.1, "test": 0.1}


def duplicate_groups(sentences: pd.DataFrame) -> pd.Series:
    """Map row_id -> group id, where rows sharing any sentence text share a group."""
    parent: dict[int, int] = {}

    def find(x: int) -> int:
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for row_ids in sentences.groupby("text").row_id.unique():
        first = find(int(row_ids[0]))
        for other in row_ids[1:]:
            root = find(int(other))
            if root != first:
                parent[root] = first
    rows = sentences.row_id.unique()
    return pd.Series([find(int(r)) for r in rows], index=rows, name="group")


def assign_splits(sentences: pd.DataFrame, strata_col: str, seed: int) -> pd.DataFrame:
    """Return one line per row_id with its stratum, duplicate group and split."""
    groups = duplicate_groups(sentences)
    rows = sentences.drop_duplicates("row_id").set_index("row_id")[[strata_col]]
    rows["group"] = groups
    rng = np.random.default_rng(seed)
    out = []
    for stratum, part in rows.groupby(strata_col, sort=True):
        # Group sizes in rows; shuffle groups, then fill train, val, test by cumulative share.
        sizes = part.groupby("group").size()
        order = sizes.index.to_numpy()[rng.permutation(len(sizes))]
        cum = sizes.loc[order].cumsum().to_numpy() / sizes.sum()
        bounds = np.cumsum(list(SPLIT_FRACTIONS.values()))
        names = np.array(list(SPLIT_FRACTIONS))
        split_of_group = dict(zip(order, names[np.searchsorted(bounds, cum - 1e-12, side="left")]))
        out.append(part.assign(split=part.group.map(split_of_group)))
    return pd.concat(out).reset_index().sort_values("row_id").reset_index(drop=True)

In [ ]:
BATCH_ROWS = 20_000
PROCESSED = WORK / "processed"


def strata_name(pair: str, romanized: bool) -> str:
    if pair == "Hindi-English":
        return "Hindi-English (romanized)" if romanized else "Hindi-English (Devanagari)"
    return pair


def romanized_rows(sentences: pd.DataFrame) -> set[int]:
    """Hindi-English rows with no Devanagari letter in any sentence (same rule as is_romanized_hindi)."""
    hindi = sentences[sentences.first_language == "Hindi"]
    has_deva = hindi.groupby("row_id").text.agg(
        lambda texts: any(char_script(c) == "devanagari" for t in texts for c in t))
    return set(has_deva.index[~has_deva])


def make_splits(ds, seed: int) -> pd.DataFrame:
    """Light pass: sentence texts only, then row-level splits stratified by pair and script."""
    parts = []
    for batch in iter_raw_batches(ds.select_columns(["row_id", "first_language", "second_language",
                                                      "data_generation_result"]), BATCH_ROWS):
        sents, _ = flatten(batch)
        roman = romanized_rows(sents)
        sents["strata"] = [strata_name(p, r in roman) for p, r in zip(sents.pair, sents.row_id)]
        parts.append(sents[["row_id", "strata", "text"]])
    sentences = pd.concat(parts, ignore_index=True)
    return assign_splits(sentences, "strata", seed=seed)[["row_id", "strata", "group", "split"]]


def process(sentences: pd.DataFrame) -> pd.DataFrame:
    """Add token, tag and label columns. Romanized Hindi is decided per source row."""
    tokens = [tokenize(t) for t in sentences.text]
    row_tokens: dict[int, list] = {}
    for row_id, toks in zip(sentences.row_id, tokens):
        row_tokens.setdefault(row_id, []).extend(toks)
    roman = {r for r, toks in row_tokens.items() if is_romanized_hindi(toks, "Hindi")}

    cols = {k: [] for k in ("tokens", "starts", "ends", "langs", "lid_source", "y_sw", "y_dur", "romanized")}
    for toks, row_id, l1, l2 in zip(tokens, sentences.row_id, sentences.first_language, sentences.second_language):
        romanized = l1 == "Hindi" and row_id in roman
        langs, sources = tag_sentence(toks, l1, l2, romanized)
        y_sw, y_dur = make_labels(langs)
        cols["tokens"].append([t.text for t in toks])
        cols["starts"].append([t.start for t in toks])
        cols["ends"].append([t.end for t in toks])
        cols["langs"].append(langs)
        cols["lid_source"].append(sources)
        cols["y_sw"].append(y_sw)
        cols["y_dur"].append(y_dur)
        cols["romanized"].append(romanized)
    out = sentences.assign(**cols)
    out["n_tokens"] = out.tokens.str.len()
    out["strata"] = [strata_name(p, r) for p, r in zip(out.pair, out.romanized)]
    return out


def batch_counts(df: pd.DataFrame) -> pd.DataFrame:
    """Per stratum and split: sentence/token counts, target positions, switches, duration bins."""
    rows = []
    for (strata, split), g in df.groupby(["strata", "split"]):
        sw = np.concatenate([np.asarray(x) for x in g.y_sw])
        dur = np.concatenate([np.asarray(x) for x in g.y_dur])
        rows.append(dict(strata=strata, split=split, sentences=len(g), tokens=int(g.n_tokens.sum()),
                         positions=int((sw != IGNORE).sum()), switches=int((sw == 1).sum()),
                         dur_1_2=int((dur == 0).sum()), dur_3_6=int((dur == 1).sum()),
                         dur_7plus=int((dur == 2).sum())))
    return pd.DataFrame(rows)


def summarize(counts: pd.DataFrame, splits: pd.DataFrame) -> pd.DataFrame:
    summ = counts.groupby(["strata", "split"]).sum().reset_index()
    rows = splits.groupby(["strata", "split"]).size().rename("rows").reset_index()
    summ = rows.merge(summ, on=["strata", "split"])
    summ["switch_rate"] = summ.switches / summ.positions
    return summ

In [ ]:
t0 = time.time()
splits = make_splits(ds, SEED)
splits.to_csv(WORK / "row_splits.csv.gz", index=False)
print(splits.split.value_counts().to_dict(), f"({time.time() - t0:.0f}s)")

split_of = splits.set_index("row_id").split
PROCESSED.mkdir(exist_ok=True)
counts = []
for i, batch in enumerate(iter_raw_batches(ds, BATCH_ROWS)):
    sents, report = flatten(batch)
    df = process(sents)
    df["split"] = df.row_id.map(split_of)
    assert df.split.notna().all()
    df.to_parquet(PROCESSED / f"part-{i:05d}.parquet", index=False)
    counts.append(batch_counts(df))
    print(f"batch {i}: {len(batch)} rows, {len(df)} sentences ({time.time() - t0:.0f}s)")

summary = summarize(pd.concat(counts), splits)
summary.to_csv(WORK / "data_summary.csv", index=False)

## 7. What the labels look like

In [ ]:
t = summary.groupby("strata")[["rows", "sentences", "positions", "switches", "dur_1_2", "dur_3_6", "dur_7plus"]].sum()
t["switch_rate"] = (t.switches / t.positions).round(3)
for c in ["dur_1_2", "dur_3_6", "dur_7plus"]:
    t[c + " %"] = (100 * t[c] / t.switches).round(1)
t[["rows", "sentences", "positions", "switch_rate", "dur_1_2 %", "dur_3_6 %", "dur_7plus %"]].sort_values("switch_rate")

In [ ]:
import matplotlib.pyplot as plt


# Reference palette (dataviz skill, light mode): surface, inks, hairlines, categorical slots,
# and the blue ordinal ramp (steps 250 / 450 / 650) for the ordered duration classes.
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SLOT1, SLOT2 = "#2a78d6", "#eb6834"
ORDINAL = ["#86b6ef", "#2a78d6", "#104281"]

plt.rcParams.update({
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"], "font.size": 10,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
})


def style_x_grid(ax) -> None:
    ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)


def title(ax, text: str, sub: str) -> None:
    ax.set_title(text, loc="left", color=INK, fontsize=12, fontweight="semibold", pad=22)
    ax.text(0, 1.02, sub, transform=ax.transAxes, color=INK2, fontsize=9, va="bottom")

In [ ]:
def fig_switch_rate(t: pd.DataFrame) -> None:
    r = t.switch_rate.sort_values()
    fig, ax = plt.subplots(figsize=(8, 6.2))
    ax.barh(r.index, r * 100, 0.7, color=SLOT1, edgecolor=SURFACE, linewidth=1.5)
    for name in [r.index[0], r.index[-1]]:
        ax.text(r[name] * 100 + 0.4, name, f"{r[name]:.1%}", va="center", color=INK, fontsize=9)
    ax.set_xlabel("Switch points among all target positions (%)")
    style_x_grid(ax)
    title(ax, "How often the next token switches language", "y_sw = 1 rate per stratum, all splits")
    fig.tight_layout()
    fig.savefig(WORK / "fig2_switch_rate.png", dpi=200)
    plt.show()


def fig_duration_mix(t: pd.DataFrame) -> None:
    d = t[["dur_1_2", "dur_3_6", "dur_7plus"]].div(t.switches, axis=0)
    d = d.sort_values("dur_1_2")
    fig, ax = plt.subplots(figsize=(8, 6.2))
    left = np.zeros(len(d))
    names = ["1-2 tokens (class 0)", "3-6 tokens (class 1)", "7+ tokens (class 2)"]
    for col, color, name in zip(d.columns, ORDINAL, names):
        ax.barh(d.index, d[col] * 100, 0.7, left=left * 100, color=color, edgecolor=SURFACE, linewidth=2, label=name)
        left += d[col].to_numpy()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Share of switch points (%)")
    style_x_grid(ax)
    ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.1), ncol=3, frameon=False, labelcolor=INK2)
    title(ax, "Length of the segment after a switch", "y_dur class mix per stratum (only defined where y_sw = 1)")
    fig.tight_layout()
    fig.savefig(WORK / "fig3_duration_mix.png", dpi=200)
    plt.show()


fig_switch_rate(t)
fig_duration_mix(t)

# Week 6: streaming loader and baselines

## 8. Streaming loader

Each sentence is encoded once with the model's own subword tokenizer. With a causal model, the output at subword *i* depends only on subwords 0..*i*, so one pass gives a prediction for every prefix. The prediction for word *t* is read at the last subword that ends before word *t+1* starts, and two checks make leakage impossible (see the docstring). The demo uses the XLM-R tokenizer (downloads about 10 MB).

In [ ]:
# ---- from src/streaming.py ----
"""Align word-level targets to a model's subwords for prefix-only (streaming) prediction.

Each sentence is encoded once with the model's own tokenizer. With a causal attention mask,
the hidden state at subword i depends only on subwords 0..i, so one forward pass yields a
prediction for every prefix. This is equivalent to feeding growing windows [x1..xt]
one at a time, and much faster.

The prediction for word t is read at the last subword that lies entirely inside the prefix,
meaning it ends at or before the start of word t+1. Punctuation typed right after word t
("sé,") is part of the prefix; whitespace before word t+1 is not, because the tokenizer
attaches it to the next word.

A position is only valid if the subwords up to that point are exactly what tokenizing the
prefix alone would produce. Two cases break this and are marked IGNORE (counted, not hidden):
- straddle: one subword covers the end of word t and the start of word t+1 (possible in
  text without spaces, such as Chinese characters or "Threads发文");
- retokenized: no subword crosses the boundary, but the tokenizer split the prefix
  differently because it saw the following characters.
Either case would let information from t+1 leak into the prediction at t.
"""


from dataclasses import dataclass, field


@dataclass
class Encoded:
    input_ids: list[int]
    sw_labels: list[int]   # per subword: y_sw at prediction points, IGNORE elsewhere
    dur_labels: list[int]  # per subword: y_dur at prediction points, IGNORE elsewhere
    pred_index: list[int]  # per word: subword index of its prediction, -1 if invalid
    invalid: dict[str, int] = field(default_factory=dict)


def prefix_ends(starts: list[int], ends: list[int], text: str) -> list[int]:
    """Character end of each word's prefix: the start of the next word (end of text for the last)."""
    return [starts[t + 1] if t + 1 < len(starts) else len(text) for t in range(len(starts))]


def encode_sentence(
    text: str,
    starts: list[int],
    ends: list[int],
    y_sw: list[int],
    y_dur: list[int],
    tokenizer,
    max_length: int = 128,
    check_prefix: bool = True,
) -> Encoded:
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=max_length)
    ids, offsets = enc["input_ids"], enc["offset_mapping"]
    # Special tokens (<s>, </s>, padding) carry no text. The unknown token does: it stands for
    # characters missing from the vocabulary (e.g. Cantonese 冇 in XGLM), so it counts as content.
    special = set(tokenizer.all_special_ids) - {tokenizer.unk_token_id}
    n_lead = 0
    while n_lead < len(ids) and ids[n_lead] in special:
        n_lead += 1
    content = [i for i in range(len(ids)) if ids[i] not in special]

    sw = [IGNORE] * len(ids)
    dur = [IGNORE] * len(ids)
    pred = [-1] * len(starts)
    invalid = {"straddle": 0, "retokenized": 0, "truncated": 0}
    bounds = prefix_ends(starts, ends, text)

    for t in range(len(starts) - 1):  # the last word has no target
        if y_sw[t] == IGNORE:
            continue
        inside = [i for i in content if offsets[i][1] <= bounds[t]]
        k = inside[-1] if inside else None
        if k is None or offsets[k][1] < ends[t]:
            # No subword ends at word t: either it was truncated away or a subword straddles the boundary.
            truncated = not any(offsets[i][0] < bounds[t] < offsets[i][1] for i in content)
            invalid["truncated" if truncated else "straddle"] += 1
            continue
        if check_prefix:
            prefix_ids = tokenizer(text[: bounds[t]].rstrip(), add_special_tokens=False)["input_ids"]
            if prefix_ids != ids[n_lead : k + 1]:
                invalid["retokenized"] += 1
                continue
        pred[t] = k
        sw[k] = y_sw[t]
        dur[k] = y_dur[t]
    return Encoded(ids, sw, dur, pred, invalid)

In [ ]:
from transformers import AutoTokenizer
xlmr_tok = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

def encode_text(text, langs, tok):
    toks = tokenize(text)
    y_sw, y_dur = make_labels(langs)
    return toks, encode_sentence(text, [t.start for t in toks], [t.end for t in toks], y_sw, y_dur, tok)

text = "Yo no sé, but I think so"
toks, e = encode_text(text, ["es", "es", "es", "en", "en", "en", "en"], xlmr_tok)
pieces = xlmr_tok.convert_ids_to_tokens(e.input_ids)
for t, k in enumerate(e.pred_index[:-1]):
    print(f"word {t} {toks[t].text!r:8} -> reads subword {k} {pieces[k]!r:8} y_sw={e.sw_labels[k]} y_dur={e.dur_labels[k]}")

# Leakage check: replacing everything after word t must not change the inputs up to its prediction point
for text, langs in [("Yo no sé, but I think so", ["es", "es", "es", "en", "en", "en", "en"]),
                    ("呢個IG update好正", ["yue", "yue", "en", "en", "yue", "yue"])]:
    toks, e = encode_text(text, langs, xlmr_tok)
    for t, k in enumerate(e.pred_index):
        if k >= 0:
            altered = text[: toks[t + 1].start] + "zzz qqq 完全不同"
            _, alt = encode_text(altered, ["x"] * len(tokenize(altered)), xlmr_tok)
            assert alt.input_ids[: k + 1] == e.input_ids[: k + 1], f"leak at word {t}"
print("no-lookahead checks passed")

## 9. Batching and rebalancing the pairs

Cantonese-English is 64% of sentences. Training draws pair *p* with probability proportional to n_p^0.3 (the exponent used for XLM-R and mT5), which cuts Cantonese to about 16% without discarding data. "Overall" metrics are macro averages over strata, so Cantonese cannot dominate the evaluation either.

In [ ]:
# ---- from src/dataset.py ----
"""PyTorch dataset, batching and pair-balanced sampling for training.

Language pairs differ a lot in size (Cantonese-English is 64% of sentences), so training
draws sentences with temperature sampling: pair p is drawn with probability proportional
to n_p ** alpha. alpha = 1 keeps the natural imbalance, alpha = 0 samples all pairs equally,
and alpha = 0.3 is the compromise used for XLM-R and mT5. Sentences within a pair are
drawn uniformly.
"""


import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, WeightedRandomSampler


def pair_sampling_probs(pairs: pd.Series, alpha: float) -> pd.Series:
    """Probability of drawing each pair under temperature sampling."""
    sizes = pairs.value_counts().astype(float)
    weights = sizes**alpha
    return weights / weights.sum()


def temperature_sampler(pairs: pd.Series, alpha: float, num_samples: int, seed: int) -> WeightedRandomSampler:
    """Sampler over sentence indices: each sentence gets P(pair) / n_pair."""
    probs = pair_sampling_probs(pairs, alpha)
    sizes = pairs.value_counts()
    weights = pairs.map(probs / sizes).to_numpy(dtype=np.float64, copy=True)
    gen = torch.Generator().manual_seed(seed)
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples, replacement=True,
                                 generator=gen)


class SwitchDataset(Dataset):
    """Encodes sentences lazily with a model's tokenizer. One item = one sentence."""

    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 128, check_prefix: bool = True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.check_prefix = check_prefix

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, i: int) -> dict:
        r = self.df.iloc[i]
        e = encode_sentence(r.text, list(r.starts), list(r.ends), list(r.y_sw), list(r.y_dur),
                            self.tokenizer, self.max_length, self.check_prefix)
        return {"input_ids": e.input_ids, "sw_labels": e.sw_labels, "dur_labels": e.dur_labels,
                "pair": r.pair, "strata": r.strata, "index": i}


def make_collate(pad_id: int):
    """Right-pad a batch. Causal models never attend to later (padded) positions."""

    def collate(items: list[dict]) -> dict:
        n = max(len(x["input_ids"]) for x in items)
        ids = np.full((len(items), n), pad_id, dtype=np.int64)
        mask = np.zeros((len(items), n), dtype=np.int64)
        sw = np.full((len(items), n), IGNORE, dtype=np.int64)
        dur = np.full((len(items), n), IGNORE, dtype=np.int64)
        for b, x in enumerate(items):
            m = len(x["input_ids"])
            ids[b, :m], mask[b, :m] = x["input_ids"], 1
            sw[b, :m], dur[b, :m] = x["sw_labels"], x["dur_labels"]
        return {"input_ids": torch.from_numpy(ids), "attention_mask": torch.from_numpy(mask),
                "sw_labels": torch.from_numpy(sw), "dur_labels": torch.from_numpy(dur),
                "pair": [x["pair"] for x in items], "strata": [x["strata"] for x in items],
                "index": torch.tensor([x["index"] for x in items])}

    return collate

In [ ]:
train_keys = pd.read_parquet(PROCESSED, columns=["pair"], filters=[("split", "==", "train")])
probs = pair_sampling_probs(train_keys.pair, alpha=0.3)
natural = train_keys.pair.value_counts(normalize=True)
pd.DataFrame({"natural share": natural.round(3), "sampled share (alpha=0.3)": probs.round(3)}).sort_values("natural share", ascending=False)

In [ ]:
# One real batch through the streaming dataset (XLM-R tokenizer), to show the shapes
demo = pd.read_parquet(PROCESSED, columns=["pair", "strata", "text", "starts", "ends", "y_sw", "y_dur"],
                       filters=[("split", "==", "val")]).head(8)
batch = make_collate(xlmr_tok.pad_token_id)([SwitchDataset(demo, xlmr_tok)[i] for i in range(len(demo))])
print({k: tuple(v.shape) for k, v in batch.items() if hasattr(v, "shape")})
print("prediction points in this batch:", int((batch["sw_labels"] != IGNORE).sum()))

## 10. Prefix-only features and evaluation

The stored tags can use the right-hand neighbor (context rule), so baseline features are rebuilt from tags visible in the prefix only. Evaluation reports, per stratum and as a macro average: switch precision, recall, F1 (the "anticipatory F1"), PR-AUC; duration accuracy and macro-F1 at true switch points; and sigma, the standard deviation of switch F1 across strata (lower = more universal).

In [ ]:
# ---- from src/features.py ----
"""Prefix-only features for the baseline models.

The stored language tags (`langs`) are labels: rule 3 of src/lid.py may fill an undecided
token from its RIGHT neighbor, which uses the future. Features must not, so they are built
from "causal tags": each token's own script/frequency decision (lid_source is script or
freq), with undecided tokens filled from the nearest decided token on the LEFT within the
prefix (or, before any decided token, from the first decided token so far; failing that
the row's first language).

Features at token t (all computed from tokens 0..t and the text typed up to word t+1):
- strata (language pair, known in advance), one-hot
- current language is the second language of the pair
- length of the current same-language run, bucketed 1, 2, 3, 4-6, 7+
- number of switches so far, bucketed 0, 1, 2, 3+
- position t, bucketed 0, 1, 2, 3-5, 6-10, 11+
- punctuation typed after word t: comma-like, sentence-final, other
- script of word t
"""


import numpy as np


DECIDED = {"script", "freq"}
COMMA = set(",，、;；:：")
FINAL = set(".!?。！？…")


def causal_tags(langs: list[str], sources: list[str], lang1: str) -> list[list[str]]:
    """causal_tags(...)[t][j] = tag of token j as seen with only tokens 0..t available."""
    raw = [lang if src in DECIDED else None for lang, src in zip(langs, sources)]
    out = []
    for t in range(len(raw)):
        prefix = raw[: t + 1]
        first = next((x for x in prefix if x is not None), lang1)
        seen, last = [], None
        for x in prefix:
            last = x if x is not None else last
            seen.append(last if last is not None else first)
        out.append(seen)
    return out


def bucket(x: int, edges: list[int]) -> int:
    """Index of the first edge >= x (edges are inclusive upper bounds), else len(edges)."""
    for i, e in enumerate(edges):
        if x <= e:
            return i
    return len(edges)


def punct_after(text: str, end: int, next_start: int) -> tuple[int, int, int]:
    gap = set(text[end:next_start]) - {" ", "\t", "\n"}
    return int(bool(gap & COMMA)), int(bool(gap & FINAL)), int(bool(gap - COMMA - FINAL))


SCRIPTS = ["latin", "han", "hiragana", "katakana", "hangul", "arabic", "devanagari", "cyrillic", "num", "other"]


def sentence_features(text, starts, ends, tokens_scripts, langs, sources, y_sw, lang1, lang2):
    """Rows of raw (unencoded) features for every target position of one sentence."""
    tags = causal_tags(langs, sources, lang1)
    rows = []
    for t in range(len(langs) - 1):
        if y_sw[t] == IGNORE:
            continue
        seen = tags[t]
        cur = seen[t]
        run = 1
        while run <= t and seen[t - run] == cur:
            run += 1
        switches = sum(seen[j] != seen[j - 1] for j in range(1, t + 1))
        comma, final, other = punct_after(text, ends[t], starts[t + 1])
        rows.append((int(cur == lang2), bucket(run, [1, 2, 3, 6]), bucket(switches, [0, 1, 2]),
                     bucket(t, [0, 1, 2, 5, 10]), comma, final, other,
                     SCRIPTS.index(tokens_scripts[t]) if tokens_scripts[t] in SCRIPTS else SCRIPTS.index("other")))
    return rows


FEATURE_NAMES = ["cur_is_lang2", "run_bucket", "switches_bucket", "pos_bucket", "punct_comma", "punct_final",
                 "punct_other", "script"]
CATEGORICAL = ["run_bucket", "switches_bucket", "pos_bucket", "script"]


def as_array(rows: list[tuple]) -> np.ndarray:
    return np.asarray(rows, dtype=np.int16).reshape(-1, len(FEATURE_NAMES))

In [ ]:
# Features must not change when only later words change
text = "Yo no sé, but I think so"
def feats(text):
    toks = tokenize(text)
    langs, sources = tag_sentence(toks, "Spanish", "English")
    y_sw, _ = make_labels(langs)
    return toks, sentence_features(text, [t.start for t in toks], [t.end for t in toks], [t.script for t in toks],
                                   langs, sources, y_sw, "Spanish", "English")
toks, base = feats(text)
for t in range(len(toks) - 1):
    assert feats(text[: toks[t + 1].start] + "completamente distinto aquí ahora")[1][t] == base[t]
print("prefix-only feature checks passed")

In [ ]:
# ---- from src/metrics.py ----
"""Evaluation shared by every model (baselines and transformers).

Inputs are flat arrays over target positions, each tagged with its stratum (language pair,
Hindi split by script). Reported per stratum and overall, where "overall" is the macro
average over strata so the large Cantonese pair cannot dominate.

y_sw:  precision, recall, F1 of the SWITCH class ("anticipatory F1"), PR-AUC, accuracy.
y_dur: accuracy and macro-F1 on positions where the TRUE y_sw = 1, plus a confusion matrix.
Universality: standard deviation of switch F1 across strata (lower = more universal).
"""


import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_recall_fscore_support


def best_threshold(y_true: np.ndarray, p_switch: np.ndarray) -> float:
    """Threshold maximizing switch F1. Choose it on training data, never on the evaluated split."""
    from sklearn.metrics import precision_recall_curve

    prec, rec, thr = precision_recall_curve(y_true, p_switch)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[int(np.argmax(f1))]) if len(thr) else 0.5


def switch_metrics(y_true: np.ndarray, p_switch: np.ndarray, threshold: float = 0.5) -> dict:
    y_pred = (p_switch >= threshold).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1], average=None, zero_division=0)
    return {
        "sw_precision": float(prec[0]), "sw_recall": float(rec[0]), "sw_f1": float(f1[0]),
        "sw_pr_auc": float(average_precision_score(y_true, p_switch)) if y_true.any() else float("nan"),
        "sw_accuracy": float((y_pred == y_true).mean()),
        "sw_positive_rate": float(y_true.mean()),
        "n_positions": int(len(y_true)),
    }


def duration_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    if len(y_true) == 0:
        return {"dur_accuracy": float("nan"), "dur_macro_f1": float("nan"), "n_switches": 0}
    return {
        "dur_accuracy": float((y_true == y_pred).mean()),
        "dur_macro_f1": float(f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
        "n_switches": int(len(y_true)),
    }


def evaluate(
    strata: np.ndarray,
    y_sw: np.ndarray,
    p_switch: np.ndarray,
    y_dur: np.ndarray,
    dur_pred: np.ndarray,
    threshold: float = 0.5,
) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    """Return (per-stratum table, overall summary, duration confusion matrix over all strata).

    All arrays are aligned over target positions (y_sw != IGNORE). dur_pred must be given at
    every position; it is only scored where y_sw == 1 and y_dur != IGNORE.
    """
    strata, y_sw, p_switch = np.asarray(strata), np.asarray(y_sw), np.asarray(p_switch)
    y_dur, dur_pred = np.asarray(y_dur), np.asarray(dur_pred)
    assert (y_sw != IGNORE).all(), "pass only target positions"

    rows = []
    for s in sorted(set(strata)):
        m = strata == s
        d = m & (y_sw == 1) & (y_dur != IGNORE)
        rows.append({"strata": s, **switch_metrics(y_sw[m], p_switch[m], threshold),
                     **duration_metrics(y_dur[d], dur_pred[d])})
    table = pd.DataFrame(rows).set_index("strata")

    metric_cols = ["sw_precision", "sw_recall", "sw_f1", "sw_pr_auc", "sw_accuracy", "dur_accuracy", "dur_macro_f1"]
    overall = {f"macro_{c}": float(table[c].mean()) for c in metric_cols}
    overall["sigma_universality_sw_f1"] = float(table.sw_f1.std(ddof=0))
    overall["sigma_universality_dur_macro_f1"] = float(table.dur_macro_f1.std(ddof=0))
    pooled = switch_metrics(y_sw, p_switch, threshold)
    overall.update({f"pooled_{k}": v for k, v in pooled.items()})

    d = (y_sw == 1) & (y_dur != IGNORE)
    cm = confusion_matrix(y_dur[d], dur_pred[d], labels=[0, 1, 2])
    cm = pd.DataFrame(cm, index=[f"true_{n}" for n in DURATION_NAMES], columns=[f"pred_{n}" for n in DURATION_NAMES])
    return table, overall, cm

## 11. Baselines

- **Majority class:** always "no switch" (probability = training switch rate) and the most frequent duration class.
- **Logistic regression** on the prefix-only features plus stratum and stratum x current language, trained on 3,000 sentences per stratum. The switch threshold is chosen on training data, never on the evaluated split. Fitted with PyTorch (the same model as scikit-learn's, which is blocked on one teammate's Windows laptop).

Evaluated on the validation split (the test split stays untouched until the final evaluation).

In [ ]:
COLUMNS = ["strata", "split", "first_language", "second_language", "text", "starts", "ends", "tokens", "langs",
           "lid_source", "y_sw", "y_dur"]


def build(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Features, strata, y_sw and y_dur over all target positions of the given sentences."""
    X, strata, ysw, ydur = [], [], [], []
    for r in df.itertuples(index=False):
        y_sw, y_dur = list(r.y_sw), list(r.y_dur)
        scripts = [char_script(tok[0]) or "other" for tok in r.tokens]
        rows = sentence_features(r.text, list(r.starts), list(r.ends), scripts, list(r.langs), list(r.lid_source),
                                 y_sw, r.first_language, r.second_language)
        X.extend(rows)
        targets = [t for t in range(len(y_sw)) if y_sw[t] != IGNORE]
        strata.extend([r.strata] * len(targets))
        ysw.extend(y_sw[t] for t in targets)
        ydur.extend(y_dur[t] for t in targets)
    return as_array(X), np.array(strata), np.array(ysw), np.array(ydur)


def design(X: np.ndarray, strata: np.ndarray) -> pd.DataFrame:
    """Feature table with stratum and a stratum x current-language interaction."""
    d = pd.DataFrame(X, columns=FEATURE_NAMES)
    d["strata"] = strata
    d["strata_x_lang"] = [f"{s}|{c}" for s, c in zip(strata, d.cur_is_lang2)]
    return d


def one_hot(d: pd.DataFrame, cat_cols: list[str], columns: list[str] | None = None) -> tuple[np.ndarray, list[str]]:
    """Dummy-encode categorical columns; reuse training `columns` so unseen levels map to all zeros."""
    z = pd.get_dummies(d[cat_cols].astype(str), dtype=np.float32)
    z = pd.concat([z, d.drop(columns=cat_cols).astype(np.float32)], axis=1)
    if columns is not None:
        z = z.reindex(columns=columns, fill_value=0.0)
    return z.to_numpy(np.float32), list(z.columns)


class LogReg:
    """Multinomial logistic regression with an L2 penalty, fitted by full-batch L-BFGS."""

    def __init__(self, l2: float = 1e-4, max_iter: int = 200, seed: int = 0):
        self.l2, self.max_iter, self.seed = l2, max_iter, seed

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LogReg":
        torch.manual_seed(self.seed)
        X, y = torch.from_numpy(X), torch.from_numpy(y.astype(np.int64))
        self.n_classes = int(y.max()) + 1
        self.layer = torch.nn.Linear(X.shape[1], self.n_classes)
        opt = torch.optim.LBFGS(self.layer.parameters(), max_iter=self.max_iter, line_search_fn="strong_wolfe")

        def closure():
            opt.zero_grad()
            loss = torch.nn.functional.cross_entropy(self.layer(X), y) + self.l2 * self.layer.weight.pow(2).sum()
            loss.backward()
            return loss

        opt.step(closure)
        return self

    @torch.no_grad()
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self.layer(torch.from_numpy(X)).softmax(-1).numpy()

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X).argmax(1)

In [ ]:
t0 = time.time()
keys = pd.read_parquet(PROCESSED, columns=["row_id", "sent_idx", "strata"], filters=[("split", "==", "train")])
picked = keys.groupby("strata").sample(3000, random_state=SEED)
train = pd.read_parquet(PROCESSED, columns=COLUMNS + ["row_id", "sent_idx"],
                        filters=[("split", "==", "train"), ("row_id", "in", sorted(set(picked.row_id)))])
train = train.merge(picked[["row_id", "sent_idx"]], on=["row_id", "sent_idx"])
evald = pd.read_parquet(PROCESSED, columns=COLUMNS, filters=[("split", "==", "val")])
Xtr, str_tr, ysw_tr, ydur_tr = build(train)
Xev, str_ev, ysw_ev, ydur_ev = build(evald)
print(f"features: train {len(ysw_tr)} positions, val {len(ysw_ev)} ({time.time() - t0:.0f}s)")

# Majority class
rate = float(ysw_tr.mean())
dur_major = int(np.bincount(ydur_tr[ysw_tr == 1], minlength=3).argmax())
maj_table, maj_overall, _ = evaluate(str_ev, ysw_ev, np.full(len(ysw_ev), rate), ydur_ev, np.full(len(ysw_ev), dur_major))

# Logistic regression
cat_cols = CATEGORICAL + ["strata", "strata_x_lang"]
Ztr, columns = one_hot(design(Xtr, str_tr), cat_cols)
Zev, _ = one_hot(design(Xev, str_ev), cat_cols, columns)
sw_model = LogReg(seed=SEED).fit(Ztr, ysw_tr)
threshold = best_threshold(ysw_tr, sw_model.predict_proba(Ztr)[:, 1])
m = ysw_tr == 1
dur_model = LogReg(seed=SEED).fit(Ztr[m], ydur_tr[m])
lr_table, lr_overall, lr_cm = evaluate(str_ev, ysw_ev, sw_model.predict_proba(Zev)[:, 1], ydur_ev,
                                       dur_model.predict(Zev), threshold=threshold)
print(f"threshold {threshold:.3f} ({time.time() - t0:.0f}s)")

cols = ["macro_sw_f1", "macro_sw_pr_auc", "macro_sw_precision", "macro_sw_recall", "sigma_universality_sw_f1",
        "macro_dur_macro_f1", "macro_dur_accuracy"]
pd.DataFrame({"majority": maj_overall, "logistic regression": lr_overall}).T[cols].round(3)

In [ ]:
lr_table[["sw_precision", "sw_recall", "sw_f1", "sw_pr_auc", "dur_accuracy", "dur_macro_f1"]].round(3).sort_values("sw_f1")

In [ ]:
print("Duration confusion matrix (rows = true class, all strata):")
lr_cm

In [ ]:
f1 = lr_table.sw_f1.sort_values()
fig, ax = plt.subplots(figsize=(8, 6.2))
ax.barh(f1.index, f1, 0.7, color=SLOT1, edgecolor=SURFACE, linewidth=1.5)
ax.axvline(f1.mean(), color=INK2, linewidth=1)
ax.text(f1.mean() + 0.005, len(f1) - 0.4, f"macro average {f1.mean():.3f}", color=INK2, fontsize=9, va="bottom")
ax.set_xlim(0, 0.8)
ax.set_xlabel("Switch F1 on the validation split")
style_x_grid(ax)
title(ax, "Logistic regression baseline: switch F1 per stratum", "Prefix-only features; the majority baseline scores 0 everywhere")
fig.tight_layout()
plt.show()

## Summary

- **Week 4:** 974,994 sentences from 15 pairs, language ID per token (script, frequency and context rules), romanized Hindi handled with a corpus-derived word list.
- **Week 5:** switch and duration labels with hand-checked tests; fixed 80/10/10 splits by conversation, stratified by pair.
- **Week 6:** prefix-only streaming loader with no-lookahead checks, pair rebalancing, and baselines. The logistic regression (switch F1 about 0.555 macro, duration macro-F1 about 0.531) is the bar the transformer models must beat from week 8.

**Open item:** hand validation of the language tags (sheets in the repository under `results/lid_check/`).